In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
model_path = "./model_out/checkpoint-1461"    # your output_dir from training

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path, local_files_only=True).to("mps")
model.eval()

/Users/eileen.zhang/contextual-live-translation/venv/lib/python3.13/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(59515, 512, padding_idx=59513)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(59515, 512, padding_idx=59513)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [ ]:
def translate_waitk(src_en, prev_fr="", k=3, max_len=80, debug=False):
    # Tokenize the source sentence
    full_tokens = tokenizer.tokenize(src_en)
    generated_ids = []
    
    # Get decoder start token ID from model config
    decoder_start_token_id = model.config.decoder_start_token_id if hasattr(model.config, 'decoder_start_token_id') else None
    if decoder_start_token_id is None:
        decoder_start_token_id = tokenizer.pad_token_id

    for t in range(max_len):
        # reveal source according to wait-k rule
        visible_src = full_tokens[: min(len(full_tokens), k + t)]
        visible_src_text = tokenizer.convert_tokens_to_string(visible_src)

        # build the input text with context (encoder input)
        if prev_fr:
            encoder_input = prev_fr + " </ctx> " + visible_src_text
        else:
            encoder_input = visible_src_text

        # Tokenize encoder input
        enc = tokenizer(
            encoder_input, return_tensors="pt", truncation=True, max_length=512
        ).to("mps")
        
        # Prepare decoder input with previously generated tokens
        # For seq2seq, we need to prepend decoder_start_token_id if this is the first token
        if len(generated_ids) == 0:
            decoder_input_ids = torch.tensor([[decoder_start_token_id]], device="mps")
        else:
            # Include decoder_start_token_id at the beginning, then generated tokens
            decoder_input_ids = torch.tensor([[decoder_start_token_id] + generated_ids], device="mps")

        # Forward pass to get next token logits
        with torch.no_grad():
            outputs = model(
                input_ids=enc["input_ids"],
                attention_mask=enc.get("attention_mask", None),
                decoder_input_ids=decoder_input_ids
            )
            
            # Get logits for the last position
            next_token_logits = outputs.logits[0, -1, :]
            
            # Apply temperature or just take argmax
            next_token_id = torch.argmax(next_token_logits, dim=-1).item()
            
            if debug and t < 10:  # Debug first few tokens
                token_text = tokenizer.decode([next_token_id], skip_special_tokens=True)
                top_k_tokens = torch.topk(next_token_logits, 3)
                top_tokens_str = ", ".join([f"{tokenizer.decode([idx.item()], skip_special_tokens=True)}({idx.item()})" 
                                           for idx in top_k_tokens.indices])
                print(f"Step {t}: token_id={next_token_id}, token='{token_text}', top3=[{top_tokens_str}], EOS={next_token_id == tokenizer.eos_token_id}, PAD={next_token_id == tokenizer.pad_token_id}")

        # Only stop on EOS, not PAD (PAD might be a valid token in some cases)
        # Also, don't stop immediately if we haven't generated at least a few tokens
        if next_token_id == tokenizer.eos_token_id:
            if debug:
                print(f"Stopping at step {t}: EOS token generated")
            break
        
        # Don't add PAD tokens to output
        if next_token_id == tokenizer.pad_token_id:
            if debug:
                print(f"Skipping PAD token at step {t}")
            continue

        generated_ids.append(next_token_id)

    if len(generated_ids) == 0:
        if debug:
            print("No tokens generated!")
        return ""
    
    result = tokenizer.decode(generated_ids, skip_special_tokens=True)
    if debug:
        print(f"Generated {len(generated_ids)} tokens, decoded: '{result}'")
    return result


In [16]:
src = "I saw him at the cafe earlier today."
ctx = "Je t'ai vu au marché hier."   # French previous sentence

# Debug: Check tokenizer properties
print(f"EOS token ID: {tokenizer.eos_token_id}")
print(f"PAD token ID: {tokenizer.pad_token_id}")
decoder_start = model.config.decoder_start_token_id if hasattr(model.config, 'decoder_start_token_id') else None
print(f"Decoder start token ID (from config): {decoder_start}")
print()

try:
    print("Running with debug=True:")
    print("-" * 50)
    translation = translate_waitk(src, ctx, k=3, debug=True)
    print("-" * 50)
    print(f"\nSource: {src}")
    print(f"Context: {ctx}")
    print(f"Translation: '{translation}'")
    print(f"Translation length: {len(translation)}")
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

EOS token ID: 0
PAD token ID: 59513
Decoder start token ID (from config): 59513

Running with debug=True:
--------------------------------------------------
Step 0: token_id=131, token='Je', EOS=False, PAD=False
Step 1: token_id=0, token='', EOS=True, PAD=False
Stopping at step 1: token_id=0
Generated 1 tokens, decoded: 'Je'
--------------------------------------------------

Source: I saw him at the cafe earlier today.
Context: Je t'ai vu au marché hier.
Translation: 'Je'
Translation length: 2
